In [1]:
import cv2
import numpy as np
import os
import random
import time
import multiprocessing
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import fftpack
from skimage.feature import local_binary_pattern

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler
from sklearn.kernel_approximation import Nystroem
from sklearn.pipeline import make_pipeline

from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score

plt.style.use('ggplot')


In [2]:
ON_KAGGLE = os.path.exists('/kaggle/input')
print("Running on Kaggle:", ON_KAGGLE)

if ON_KAGGLE:
    AIREAL_AI_SRC    = '/kaggle/input/ai-real-art-dataset/AiArtData/AiArtData'
    AIREAL_REAL_SRC  = '/kaggle/input/ai-real-art-dataset/RealArt/RealArt'
    CONTENT_AI_SRC   = '/kaggle/input/content-fake-real-dataset/fake_images'
    CONTENT_REAL_SRC = '/kaggle/input/content-fake-real-dataset/real_images'
    WORKING_DIR      = '/kaggle/working'
else:
    AIREAL_AI_SRC    = r'E:\M.C.A\Project\AI-Real\AiArtData\AiArtData'
    AIREAL_REAL_SRC  = r'E:\M.C.A\Project\AI-Real\RealArt\RealArt'
    CONTENT_AI_SRC   = r'E:\M.C.A\Project\content\fake_images'
    CONTENT_REAL_SRC = r'E:\M.C.A\Project\content\real_images'
    try:
        WORKING_DIR = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        WORKING_DIR = os.getcwd()

PKL_PATH = os.path.join(WORKING_DIR, 'fdav5_enhanced.pkl')

_FFT_RESIZE = 256
_LOG_EVERY  = 2000
MAX_PER_CLASS = 10_000_000   
_N_WORKERS = max(1, multiprocessing.cpu_count() - 1)

print(f"Workers: {_N_WORKERS}  |  Max per class: {MAX_PER_CLASS:,}")


Running on Kaggle: False
Workers: 15  |  Max per class: 10,000,000


In [3]:
# Precomputed constants
_r_index_cache = {}
_INTERP_XI = np.linspace(0, 80, num=80)
_BANDS = [(0, 10), (10, 20), (20, 35), (35, 55), (55, 80)]
_X_LOG = np.log(np.arange(1, 81, dtype=np.float64))
_X_LOG_MEAN = _X_LOG.mean()
_X_LOG_DENOM = ((_X_LOG - _X_LOG_MEAN) ** 2).sum()
_X_LOG_C = _X_LOG - _X_LOG_MEAN

def _get_r_index(size):
    if size not in _r_index_cache:
        y, x = np.indices((size, size))
        c = (size - 1) / 2.0
        r = np.hypot(x - c, y - c).astype(np.int32)
        _r_index_cache[size] = r
    return _r_index_cache[size]

def azimuthalAverage_fast(image):
    r = _get_r_index(image.shape[0])
    tbin = np.bincount(r.ravel(), weights=image.ravel())
    nr = np.bincount(r.ravel())
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(nr > 0, tbin / nr, 0.0)

def _process_single_image(filepath):
    """
    Extracts 114-dimensional feature vector.
    """
    try:
        img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
        if img is None: return None
        if img.shape[0] != _FFT_RESIZE or img.shape[1] != _FFT_RESIZE:
            img = cv2.resize(img, (_FFT_RESIZE, _FFT_RESIZE), interpolation=cv2.INTER_AREA)

        # 1. PSD (80 dims)
        f = np.fft.fft2(img)
        fshift = np.fft.fftshift(f) + 1e-8
        mag = 20 * np.log10(np.abs(fshift))
        psd1D = azimuthalAverage_fast(mag)
        
        src_x = np.linspace(0, 80, num=psd1D.size)
        psd_feat = np.interp(_INTERP_XI, src_x, psd1D)
        psd_feat = psd_feat / (psd_feat.mean() + 1e-8)  # Better normalization

        # 2. Extra PSD Stats (8 dims)
        psd_sq = psd_feat ** 2
        band_energies = np.array([psd_sq[s:e].mean() for s, e in _BANDS])
        total_energy = band_energies.sum() + 1e-8
        band_ratios = band_energies / total_energy
        hf_ratio = band_energies[3:].sum() / total_energy

        y_log = np.log(np.abs(psd_feat) + 1e-8)
        slope = (_X_LOG_C * (y_log - y_log.mean())).sum() / _X_LOG_DENOM

        p_abs = np.abs(psd_feat)
        p = p_abs / (p_abs.sum() + 1e-8)
        spec_ent = -(p * np.log(p + 1e-8)).sum()
        
        stats = np.concatenate([band_ratios, [hf_ratio, slope, spec_ent]])

        # 3. DCT Energy (16 dims)
        dct = fftpack.dct(fftpack.dct(img.T, norm='ortho').T, norm='ortho')
        dct_abs = np.abs(dct)
        dct_feat = []
        step = _FFT_RESIZE // 4
        for i in range(4):
            for j in range(4):
                dct_feat.append(np.mean(dct_abs[i*step:(i+1)*step, j*step:(j+1)*step]))
        dct_feat = np.array(dct_feat)
        dct_feat = dct_feat / (dct_feat.mean() + 1e-8)

        # 4. LBP Texture (10 dims)
        lbp = local_binary_pattern(img, P=8, R=1, method='uniform')
        lbp_hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10))
        lbp_hist = lbp_hist.astype(np.float32)
        lbp_hist = lbp_hist / (lbp_hist.sum() + 1e-8)

        return np.concatenate([psd_feat, dct_feat, lbp_hist, stats]).astype(np.float32)

    except Exception:
        return None

def process_dataset_parallel(img_files, label, max_images, n_workers=_N_WORKERS):
    from multiprocessing.dummy import Pool as ThreadPool
    limit = min(max_images, len(img_files))
    files_to_process = img_files[:limit]
    
    print(f"  Processing {limit:,} images for label={label} using {n_workers} worker(s)...")
    t0 = time.time()
    
    rows, skipped = [], 0
    chunk = max(64, limit // (n_workers * 4))
    
    # Safe multiprocessing guard
    try:
        with ThreadPool(processes=n_workers) as pool:
            for i, result in enumerate(pool.imap_unordered(_process_single_image, files_to_process), start=1):
                if result is not None:
                    rows.append(result)
                else:
                    skipped += 1
                if i % _LOG_EVERY == 0:
                    elapsed = time.time() - t0
                    rate = i / elapsed if elapsed > 0 else 0
                    print(f"    [{i:,}/{limit:,}] loaded={len(rows):,} skipped={skipped:,} speed={rate:.0f} img/s")
    except Exception as e:
        print(f"  [WARNING] Parallel pool failed ({e}), falling back to single-process.")
        for fp in files_to_process:
            r = _process_single_image(fp)
            if r is not None: rows.append(r)
            else: skipped += 1

    print(f"  Done - {len(rows):,} features in {time.time() - t0:.1f}s")
    return np.array(rows, dtype=np.float32)


In [3]:
if os.path.exists(PKL_PATH):
    print(f"Found enhanced cache at {PKL_PATH}. Loading...")
    with open(PKL_PATH, 'rb') as f:
        data = pickle.load(f)
    X_total, y_total = data['data'], data['label']
    print(f"Loaded {len(y_total):,} samples with {X_total.shape[1]} dimensions.")
else:
    print("Collecting images for extraction...")
    def collect_images(src_dir, rec=False):
        if not os.path.isdir(src_dir): return []
        paths = []
        if rec:
            for r, _, fs in os.walk(src_dir):
                for f in fs:
                    if f.lower().endswith(('.jpg', '.jpeg', '.png')): paths.append(os.path.join(r, f))
        else:
            paths = [os.path.join(src_dir, f) for f in os.listdir(src_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        return sorted(paths)

    fake_files = collect_images(AIREAL_AI_SRC, False) + collect_images(CONTENT_AI_SRC, True)
    real_files = collect_images(AIREAL_REAL_SRC, False) + collect_images(CONTENT_REAL_SRC, False)
    
    random.seed(42)
    random.shuffle(fake_files)
    random.shuffle(real_files)
    
    print("\nStarting enhanced feature extraction...")
    psd1D_fake = process_dataset_parallel(fake_files, label=1, max_images=MAX_PER_CLASS)
    psd1D_real = process_dataset_parallel(real_files, label=0, max_images=MAX_PER_CLASS)
    
    label_fake = np.ones(len(psd1D_fake), dtype=np.float32)
    label_real = np.zeros(len(psd1D_real), dtype=np.float32)
    
    X_total = np.concatenate((psd1D_fake, psd1D_real), axis=0)
    y_total = np.concatenate((label_fake, label_real), axis=0)
    
    with open(PKL_PATH, 'wb') as out:
        pickle.dump({'data': X_total, 'label': y_total}, out, protocol=4)
    print(f"Cached {len(y_total)} samples of {X_total.shape[1]} dims to {PKL_PATH}")


Found enhanced cache at e:\M.C.A\Project\backend\fdav5_enhanced.pkl. Loading...
Loaded 407,877 samples with 114 dimensions.


In [ ]:
def _compute_fft_magnitude(img):
    """Computes the centered 2D FFT magnitude spectrum of an image."""
    f = np.fft.fft2(img)
    fshift = np.fft.fftshift(f) + 1e-8
    mag = 20 * np.log10(np.abs(fshift))
    return mag

def plot_single_fft_comparison():
    print("\nVisualizing FFT comparison of single images...")
    
    def get_sample_image(src_dir):
        if not os.path.isdir(src_dir): return None
        for r, _, fs in os.walk(src_dir):
            for f in fs:
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    return os.path.join(r, f)
        return None

    fake_sample = get_sample_image(AIREAL_AI_SRC) or get_sample_image(CONTENT_AI_SRC)
    real_sample = get_sample_image(AIREAL_REAL_SRC) or get_sample_image(CONTENT_REAL_SRC)

    if not fake_sample or not real_sample:
        print("[WARNING] Could not find sample images for FFT comparison.")
        return

    fake_img = cv2.imread(fake_sample, cv2.IMREAD_GRAYSCALE)
    real_img = cv2.imread(real_sample, cv2.IMREAD_GRAYSCALE)
    
    if fake_img is None or real_img is None:
        print("[WARNING] Could not load sample images for FFT comparison.")
        return
        
    fake_fft = _compute_fft_magnitude(fake_img)
    real_fft = _compute_fft_magnitude(real_img)
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes[0, 0].imshow(fake_img, cmap='gray'); axes[0, 0].set_title('AI (Fake) Image');        axes[0, 0].axis('off')
    axes[0, 1].imshow(fake_fft, cmap='gray'); axes[0, 1].set_title('FFT Magnitude - AI');     axes[0, 1].axis('off')
    axes[1, 0].imshow(real_img, cmap='gray'); axes[1, 0].set_title('Real Image');             axes[1, 0].axis('off')
    axes[1, 1].imshow(real_fft, cmap='gray'); axes[1, 1].set_title('FFT Magnitude - Real');   axes[1, 1].axis('off')
    
    plt.suptitle('FFT Comparison: AI-Generated vs Real Images', fontsize=16, fontweight='bold')
    plt.tight_layout()
    viz_path = os.path.join(WORKING_DIR, 'fft_comparison.png')
    plt.savefig(viz_path, dpi=150)
    plt.close()
    print(f"Saved FFT comparison to {viz_path}")
    
plot_single_fft_comparison()



Visualizing FFT comparison of single images...
Saved FFT comparison to e:\M.C.A\Project\backend\fft_comparison.png


In [4]:
print(f"\n=== Model Evaluation (Enhanced Features) ===")

rng = np.random.RandomState(42)
idx_ai = np.where(y_total == 1)[0]
idx_real = np.where(y_total == 0)[0]
per_class = min(MAX_PER_CLASS, len(idx_ai), len(idx_real))

idx_balanced = np.concatenate([
    rng.choice(idx_ai, size=per_class, replace=False),
    rng.choice(idx_real, size=per_class, replace=False)
])
rng.shuffle(idx_balanced)

X_eval = X_total[idx_balanced]
y_eval = y_total[idx_balanced]

X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
    X_eval, y_eval, test_size=0.2, random_state=42, stratify=y_eval
)

scaler = RobustScaler()
X_tr = scaler.fit_transform(X_tr_raw)
X_te = scaler.transform(X_te_raw)

classifiers = {
    'LinearSVC': LinearSVC(C=0.1, class_weight='balanced', max_iter=3000, dual=False),
    'SVM-RBF (Approx)': make_pipeline(
        Nystroem(kernel='rbf', gamma=None, n_components=500, random_state=42),
        LinearSVC(C=1.0, class_weight='balanced', max_iter=3000, dual=False)
    ),
    'SVM-Poly (Approx)': make_pipeline(
        Nystroem(kernel='poly', degree=3, gamma=None, n_components=500, random_state=42),
        LinearSVC(C=1.0, class_weight='balanced', max_iter=3000, dual=False)
    ),
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
}

class_names = ['Real (0)', 'AI/Fake (1)']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (name, clf) in enumerate(classifiers.items()):
    t0 = time.time()
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    
    acc = (y_pred == y_te).mean()
    bal = balanced_accuracy_score(y_te, y_pred)
    
    print(f"\n{'='*55}\n{name} | Acc: {acc:.4f} | BAcc: {bal:.4f} | Time: {time.time()-t0:.1f}s\n{'='*55}")
    print(classification_report(y_te, y_pred, target_names=class_names))

    cm = confusion_matrix(y_te, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=axes[idx])
    axes[idx].set_title(f'{name}\nAcc={acc:.3f} | BAcc={bal:.3f}', fontsize=12)
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.suptitle('Confusion Matrices — FDA', fontsize=16, fontweight='bold')
plt.tight_layout()

cm_path = os.path.join(WORKING_DIR, 'confusion_matrices_v5.png')
plt.savefig(cm_path, dpi=150)
plt.close()

print(f"\nSaved confusion matrices to {cm_path}")



=== Model Evaluation (Enhanced Features) ===

LinearSVC | Acc: 0.6499 | BAcc: 0.6499 | Time: 43.1s
              precision    recall  f1-score   support

    Real (0)       0.66      0.62      0.64     31333
 AI/Fake (1)       0.64      0.68      0.66     31332

    accuracy                           0.65     62665
   macro avg       0.65      0.65      0.65     62665
weighted avg       0.65      0.65      0.65     62665


SVM-RBF (Approx) | Acc: 0.6754 | BAcc: 0.6754 | Time: 158.9s
              precision    recall  f1-score   support

    Real (0)       0.68      0.67      0.67     31333
 AI/Fake (1)       0.67      0.68      0.68     31332

    accuracy                           0.68     62665
   macro avg       0.68      0.68      0.68     62665
weighted avg       0.68      0.68      0.68     62665


SVM-Poly (Approx) | Acc: 0.6733 | BAcc: 0.6733 | Time: 1522.3s
              precision    recall  f1-score   support

    Real (0)       0.67      0.67      0.67     31333
 AI/Fake (1